In [1]:
# =============================================================================
# Deliverable 2 — PADS Baseline Classifiers
# CS 8674 Part II · Intelligent IoT Frameworks for Chronic Disease Management
#
# Trains SVM, Random Forest, and 1D-CNN on PADS features produced by the
# D2 Dataset Pipeline notebook. Uses GroupKFold CV (subject-level) to
# prevent leakage. Reports macro-F1 and AUROC for comparison against the
# D2 Transformer.
# =============================================================================

import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------------------------------------------------------------------------
# 0. Load cleaned data from pipeline output
# ---------------------------------------------------------------------------
OUT_DIR = Path("/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/cleaned_d2")

feat_df = pd.read_parquet(OUT_DIR / "pads_all.parquet")
raw     = np.load(OUT_DIR / "pads_raw_windows.npz")
X_raw   = raw["X"]       # (N, samples, 6)
y_raw   = raw["y"]
subj_raw = raw["subject"]

feat_cols = [c for c in feat_df.columns
             if any(c.endswith(f) for f in
                    ["mean","std","rms","range","dom_freq","band_power","tremor_index"])]

X     = feat_df[feat_cols].values.astype(np.float32)
y     = feat_df["label"].values
groups = feat_df["subject_id"].values

print(f"Feature matrix : {X.shape}")
print(f"Raw windows    : {X_raw.shape}")
print(f"Subjects       : {feat_df['subject_id'].nunique()}")
print(f"PD rate        : {y.mean():.3f}")
print(f"Features/window: {X.shape[1]}")

Feature matrix : (7810, 42)
Raw windows    : (7810, 974, 6)
Subjects       : 355
PD rate        : 0.777
Features/window: 42


In [2]:
from pathlib import Path

# Search for the file in the attached inputs
found_paths = list(Path("/kaggle/input").rglob("pads_all.parquet"))

if found_paths:
    print("Found it! Copy the exact path below:")
    print(found_paths[0])
else:
    print("The file 'pads_all.parquet' could not be found anywhere under /kaggle/input.")
    print("Double check your Data pane on the right to see if the item has a green checkmark or if it's still loading.")

Found it! Copy the exact path below:
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/cleaned_d2/pads_all.parquet


In [3]:
# ---------------------------------------------------------------------------
# 1. GroupKFold evaluation helper (identical to D1)
# ---------------------------------------------------------------------------
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

N_SPLITS = 5

def groupkfold_eval(X, y, groups, model, n_splits=N_SPLITS):
    gkf = StratifiedGroupKFold(n_splits=n_splits)
    f1s, aurocs, cms = [], [], []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)

        model.fit(X_tr, y_tr)
        y_pred  = model.predict(X_te)
        y_score = model.predict_proba(X_te)[:, 1]

        f1s.append(f1_score(y_te, y_pred, average="macro"))
        aurocs.append(roc_auc_score(y_te, y_score))
        cms.append(confusion_matrix(y_te, y_pred))

    mean_cm = np.mean(cms, axis=0).astype(int)
    return {
        "macro_f1_mean":  round(float(np.mean(f1s)),  3),
        "macro_f1_std":   round(float(np.std(f1s)),   3),
        "auroc_mean":     round(float(np.mean(aurocs)),3),
        "auroc_std":      round(float(np.std(aurocs)), 3),
        "n_folds":        n_splits,
        "confusion_matrix": mean_cm,
    }

In [4]:
# ---------------------------------------------------------------------------
# 2. SVM
# ---------------------------------------------------------------------------
from sklearn.svm import SVC

print("Training SVM...")
svm = SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE)
svm_res = groupkfold_eval(X, y, groups, svm)
print(f"SVM  — macro-F1: {svm_res['macro_f1_mean']} ± {svm_res['macro_f1_std']}"
      f"   AUROC: {svm_res['auroc_mean']} ± {svm_res['auroc_std']}")

Training SVM...
SVM  — macro-F1: 0.564 ± 0.023   AUROC: 0.693 ± 0.021


In [5]:
# ---------------------------------------------------------------------------
# 3. Random Forest
# ---------------------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=400, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
rf_res = groupkfold_eval(X, y, groups, rf)
print(f"RF   — macro-F1: {rf_res['macro_f1_mean']} ± {rf_res['macro_f1_std']}"
      f"   AUROC: {rf_res['auroc_mean']} ± {rf_res['auroc_std']}")

Training Random Forest...
RF   — macro-F1: 0.498 ± 0.011   AUROC: 0.726 ± 0.02


In [6]:
# ---------------------------------------------------------------------------
# 4. 1D-CNN (raw windows)
# ---------------------------------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS  = 40
BATCH   = 64
LR      = 1e-3

print(f"Device: {DEVICE}")

class CNN1D(nn.Module):
    def __init__(self, n_channels=6, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.head(self.net(x).squeeze(-1))

def train_eval_cnn(X_raw, y_raw, subj_raw, n_splits=N_SPLITS):
    # X_raw: (N, T, C) → need (N, C, T) for Conv1d
    X_t = torch.tensor(X_raw.transpose(0, 2, 1), dtype=torch.float32)
    y_t = torch.tensor(y_raw, dtype=torch.long)

    gkf  = StratifiedGroupKFold(n_splits=n_splits)
    f1s, aurocs, cms = [], [], []

    # class weights for imbalance
    counts   = np.bincount(y_raw)
    weights  = torch.tensor(len(y_raw) / (len(counts) * counts), dtype=torch.float32).to(DEVICE)

    for fold, (tr, te) in enumerate(gkf.split(X_t, y_t, subj_raw)):
        model     = CNN1D(n_channels=X_t.shape[1]).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss(weight=weights)

        train_ds  = TensorDataset(X_t[tr], y_t[tr])
        train_dl  = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

        # Train
        model.train()
        for epoch in range(EPOCHS):
            for xb, yb in train_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        with torch.no_grad():
            X_te = X_t[te].to(DEVICE)
            logits = model(X_te).cpu()
            probs  = torch.softmax(logits, dim=1)[:, 1].numpy()
            preds  = logits.argmax(dim=1).numpy()

        y_te = y_t[te].numpy()
        f1s.append(f1_score(y_te, preds, average="macro"))
        aurocs.append(roc_auc_score(y_te, probs))
        cms.append(confusion_matrix(y_te, preds))
        print(f"  fold {fold+1}/{n_splits} — F1={f1s[-1]:.3f}  AUROC={aurocs[-1]:.3f}")

    mean_cm = np.mean(cms, axis=0).astype(int)
    return {
        "macro_f1_mean":  round(float(np.mean(f1s)),  3),
        "macro_f1_std":   round(float(np.std(f1s)),   3),
        "auroc_mean":     round(float(np.mean(aurocs)),3),
        "auroc_std":      round(float(np.std(aurocs)), 3),
        "n_folds":        n_splits,
        "confusion_matrix": mean_cm,
    }

print("Training 1D-CNN...")
cnn_res = train_eval_cnn(X_raw, y_raw, subj_raw)
print(f"CNN  — macro-F1: {cnn_res['macro_f1_mean']} ± {cnn_res['macro_f1_std']}"
      f"   AUROC: {cnn_res['auroc_mean']} ± {cnn_res['auroc_std']}")

Device: cuda
Training 1D-CNN...
  fold 1/5 — F1=0.592  AUROC=0.680
  fold 2/5 — F1=0.548  AUROC=0.742
  fold 3/5 — F1=0.557  AUROC=0.688
  fold 4/5 — F1=0.540  AUROC=0.710
  fold 5/5 — F1=0.590  AUROC=0.689
CNN  — macro-F1: 0.565 ± 0.022   AUROC: 0.702 ± 0.022


In [7]:
# ---------------------------------------------------------------------------
# 5. Results table + confusion matrices
# ---------------------------------------------------------------------------
EDA_DIR = Path("/kaggle/working/eda_d2")
EDA_DIR.mkdir(parents=True, exist_ok=True)

results = [
    {"model": "SVM",          **{k:v for k,v in svm_res.items() if k != "confusion_matrix"}},
    {"model": "RandomForest", **{k:v for k,v in rf_res.items()  if k != "confusion_matrix"}},
    {"model": "CNN1D",        **{k:v for k,v in cnn_res.items() if k != "confusion_matrix"}},
]
results_df = pd.DataFrame(results)
print("\n=== PADS Baseline Results ===")
print(results_df.to_string(index=False))

results_df.to_json(
    Path("/kaggle/working/pads_baseline_metrics.json"),
    orient="records", indent=2
)
print("\nSaved -> pads_baseline_metrics.json")

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (res, name) in zip(axes, [
    (svm_res, "SVM"), (rf_res, "Random Forest"), (cnn_res, "1D-CNN")
]):
    cm = res["confusion_matrix"]
    sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="Blues",
                xticklabels=["HC","PD"], yticklabels=["HC","PD"])
    ax.set_title(f"{name}\nF1={res['macro_f1_mean']}±{res['macro_f1_std']}"
                 f"  AUROC={res['auroc_mean']}±{res['auroc_std']}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle("PADS Baseline Classifiers — PD vs HC", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(EDA_DIR / "pads_confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved -> pads_confusion_matrices.png")


=== PADS Baseline Results ===
       model  macro_f1_mean  macro_f1_std  auroc_mean  auroc_std  n_folds
         SVM          0.564         0.023       0.693      0.021        5
RandomForest          0.498         0.011       0.726      0.020        5
       CNN1D          0.565         0.022       0.702      0.022        5

Saved -> pads_baseline_metrics.json
Saved -> pads_confusion_matrices.png


In [8]:
# ---------------------------------------------------------------------------
# 6. Feature importance (Random Forest)
# ---------------------------------------------------------------------------
import re

# Refit RF on full data for feature importance plot
scaler_full = StandardScaler()
X_scaled    = scaler_full.fit_transform(X)
rf_full     = RandomForestClassifier(
    n_estimators=400, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
rf_full.fit(X_scaled, y)

importances = pd.Series(rf_full.feature_importances_, index=feat_cols)
top20       = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(8, 6))
top20.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_title("PADS Random Forest — Top 20 Feature Importances")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.savefig(EDA_DIR / "pads_rf_feature_importance.png", dpi=120)
plt.close()
print("Saved -> pads_rf_feature_importance.png")
print("\nTop 10 features:")
print(top20.head(10).round(4).to_string())

Saved -> pads_rf_feature_importance.png

Top 10 features:
Gyro_X_tremor_index    0.0398
Gyro_Y_tremor_index    0.0369
Acc_Z_dom_freq         0.0349
Acc_X_dom_freq         0.0345
Gyro_X_dom_freq        0.0323
Acc_X_tremor_index     0.0299
Acc_Y_tremor_index     0.0289
Gyro_Y_dom_freq        0.0278
Acc_Y_dom_freq         0.0277
Gyro_X_band_power      0.0269


In [9]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/__results__.html
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/eda_summary_d2.txt
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/__notebook__.ipynb
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/__output__.json
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/custom.css
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/eda_d2/pads_windows_per_subject.png
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/eda_d2/pads_dom_freq.png
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/eda_d2/pads_label_balance.png
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/eda_d2/pads_windows_per_task.png
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/cleaned_d2/pads_train.parquet
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/cleaned_d2/pads_test.parquet
/kaggle/input/notebooks/aqn96kag/pd-glove-d2-pads-pipeline/cleaned_d2/pads_val.parquet